# The X-ray Variable Stellar (Galactic) Population

### STATiX Variable Source Catalogue — Pouliasis et al. (A&A)

This notebook reproduces the Hertzsprung-Russell (HR) diagram analysis and SIMBAD/NED classification breakdown of the Galactic STATiX variable sources described in the paper.


## Important note on catalogue columns

The **published** table (`STATIX_VC_published.fits`) does not carry pre-joined Gaia photometry — only the public cross-match identifiers `GDR3_ID` (Gaia DR3 `source_id`) and the optical counterpart coordinates `RA_OPT`, `DEC_OPT`.

This notebook therefore **queries Gaia DR3 live** to reconstruct the information needed for the HR diagram and classification plots, using only identifiers already present in `out`. This makes the notebook fully reproducible from the released catalogue alone.

**Runtime warning:** the Gaia bulk query is fast (a single TAP upload-join query). Intermediate results are cached to disk so the notebook can be safely re-run/resumed.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from collections import Counter

from astropy.table import Table, unique
from astropy.coordinates import SkyCoord
import astropy.units as u

from astroquery.gaia import Gaia
from astroquery.simbad import Simbad
from astroquery.ipac.ned import Ned

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

CACHE_DIR = Path("cache_galactic")
CACHE_DIR.mkdir(exist_ok=True)

## 1. Load the published catalogue and select Galactic sources

In [ ]:
import os

path = "STATIX_VC_published.fits"
print("File exists:", os.path.exists(path))
print("Last modified:", time.ctime(os.path.getmtime(path)))
print("File size (bytes):", os.path.getsize(path))

In [ ]:
out = Table.read(path)
print(f"Loaded {len(out)} detections.")
print("GDR3_ID dtype:", out["GDR3_ID"].dtype)  # should print int64


unique_out = unique(out, keys="group_id")
print(f"Unique sources: {len(unique_out)}")

In [ ]:
out.colnames

In [ ]:
out

In [ ]:
unique_out["final_class"]=unique_out["CLASS"]
unique_out["final_class"][unique_out["final_class"]==-1] = unique_out["CLASS_RF"][unique_out["final_class"]==-1]

In [ ]:
out["final_class"]=out["CLASS"]
out["final_class"][out["final_class"]==-1] = out["CLASS_RF"][out["final_class"]==-1]

In [ ]:
stars= unique_out[np.logical_or(unique_out["final_class"]==1,unique_out["final_class"]==0)]
len(stars)

In [ ]:
import numpy as np

unique_out["final_class"] = np.where(
    unique_out["CLASS"] == -1,
    unique_out["CLASS_RF"],
    unique_out["CLASS"]
)

stars= unique_out[np.logical_or(unique_out["final_class"]==1,unique_out["final_class"]==0)]
len(stars)

In [ ]:
# Galactic = CLASS_RF 0 (secure) or 1 (likely)
gal = unique_out[(unique_out["final_class"] == 0) | (unique_out["final_class"] == 1) ]

n_secure = np.sum(gal["final_class"] == 0)
n_likely = np.sum(gal["final_class"] == 1)

print(f"Unique Galactic sources : {len(gal)}")
print(f"  secure Galactic (0)   : {n_secure}")
print(f"  likely Galactic (1)   : {n_likely}")
print(f"Fraction of full catalogue: {len(gal)/len(unique_out):.1%}")
print()
print("Paper values to compare against: 9,903 secure + 4,615 likely; ~60% of the catalogue.")

### Sources with a reliable Gaia (GDR3) counterpart

In [ ]:
gal

In [ ]:
has_gdr3 = ~gal["GDR3_ID"].mask if hasattr(gal["GDR3_ID"], "mask") else (gal["GDR3_ID"] > 0)
gal_gaia = gal[has_gdr3]

print(f"Galactic sources with a GDR3 counterpart: {len(gal_gaia)}")


## 2. Retrieve Gaia DR3 photometry and parallaxes

We use a single bulk TAP upload-join query against `gaiadr3.gaia_source`, matching on `GDR3_ID` (the Gaia `source_id`). This is far faster than querying source-by-source.

In [ ]:
import os
#os.remove("./gaia_photometry.csv")
gaia_cache = "gaia_photometry.csv"

if os.path.exists(gaia_cache):
    gaia_df = pd.read_csv(gaia_cache)
    gaia_df.columns = [c.upper() if c.lower() == "detid" else c for c in gaia_df.columns]
    print(f"Loaded cached Gaia photometry for {len(gaia_df)} sources.")
else:
    gal_gaia_pd = gal_gaia["DETID", "GDR3_ID"].to_pandas()
    gal_gaia_pd = gal_gaia_pd.rename(columns={"GDR3_ID": "SOURCE_ID"})
    gal_gaia_pd = gal_gaia_pd[gal_gaia_pd["SOURCE_ID"].notna() & (gal_gaia_pd["SOURCE_ID"] > 0)]
    gal_gaia_pd["source_id"] = gal_gaia_pd["SOURCE_ID"].astype(np.int64)

    source_ids = gal_gaia_pd["SOURCE_ID"].tolist()

    batch_size = 500
    all_results = []
    failed_batches = []

    for i in tqdm(range(0, len(source_ids), batch_size), desc="Querying Gaia in batches"):
        batch = source_ids[i:i + batch_size]
        id_list = ",".join(str(s) for s in batch)

        query = f"""
        SELECT
            g.source_id,
            g.phot_g_mean_mag,
            g.phot_bp_mean_mag,
            g.phot_rp_mean_mag,
            g.parallax,
            g.parallax_over_error,
            g.ruwe,
            g.pmra_pmdec_corr,
            g.phot_variable_flag,
            g.classprob_dsc_combmod_quasar,
            g.classprob_dsc_combmod_galaxy,
            g.classprob_dsc_combmod_star,
            ap.classprob_dsc_combmod_whitedwarf,
            ap.classprob_dsc_combmod_binarystar
        FROM gaiadr3.gaia_source AS g
        LEFT JOIN gaiadr3.astrophysical_parameters AS ap
            ON g.source_id = ap.source_id
        WHERE g.source_id IN ({id_list})
        """

        success = False
        for attempt in range(4):
            try:
                job = Gaia.launch_job_async(query)
                batch_df = job.get_results().to_pandas()
                all_results.append(batch_df)
                success = True
                break
            except Exception as e:
                wait = 5 * (attempt + 1)
                print(f"Batch {i//batch_size}: attempt {attempt+1} failed ({e}); retrying in {wait}s...")
                time.sleep(wait)

        if not success:
            failed_batches.append(i // batch_size)
            print(f"Batch {i//batch_size}: FAILED after all retries, skipping.")

    gaia_df_raw = pd.concat(all_results, ignore_index=True)

    # --- FIX: dedup on source_id, then merge (not dict-map) to correctly
    # preserve every DETID, including cases where multiple DETIDs share
    # the same Gaia source_id. ---
    n_dup = gaia_df_raw["SOURCE_ID"].duplicated().sum()
    if n_dup > 0:
        print(f"[INFO] {n_dup} duplicate source_id rows in raw query results; deduplicating.")
    gaia_df_dedup = gaia_df_raw.drop_duplicates(subset="SOURCE_ID", keep="first")

    gaia_df = gal_gaia_pd.merge(gaia_df_dedup, on="SOURCE_ID", how="left")

    if failed_batches:
        print(f"\n[WARNING] {len(failed_batches)} batch(es) still failed after retries: {failed_batches}")
        print("Consider re-running those specific batches manually, or increasing retry attempts/backoff.")

    gaia_df.to_csv(gaia_cache, index=False)
    print(f"Retrieved Gaia photometry+astrophysical parameters for {len(gaia_df)} / {len(gal_gaia)} sources.")

In [ ]:
# Merge Gaia photometry back onto the Galactic source table.
# Join on DETID (not source_id) since gaia_df already carries DETID from the
# TAP upload join, which avoids row fan-out when several STATiX detections
# share the same Gaia counterpart.
gal_df = gal_gaia["DETID", "group_id", "GDR3_ID", "RA_OPT", "DEC_OPT", "CLASS_RF","Simbad_ID","Simbad_class","NED_ID","NED_class"].to_pandas()
gal_df = gal_df.merge(gaia_df, on="DETID", how="inner")

gal_df["bp_rp"] = gal_df["phot_bp_mean_mag"] - gal_df["phot_rp_mean_mag"]
gal_df["M_G"] = gal_df["phot_g_mean_mag"] + 5 * np.log10(gal_df["parallax"] / 1000.0) + 5

n_wd = gal_df["classprob_dsc_combmod_whitedwarf"].notna().sum()
n_bs = gal_df["classprob_dsc_combmod_binarystar"].notna().sum()

print(f"Sources with valid Gaia photometry+astrometry merged: {len(gal_df)}")
print(f"  with white-dwarf classification probability : {n_wd}")
print(f"  with binary-star classification probability  : {n_bs}")
gal_df.head()

## 3. Cross-match with SIMBAD

We query SIMBAD by Gaia DR3 identifier (`"Gaia DR3 <source_id>"`), in batches, requesting the object type (`OTYPE`).

## 5. Spectral-type classification from BP−RP

Sources are binned into approximate spectral types using their Gaia BP−RP color, following the same ranges used in the paper's HR diagram figure.

In [ ]:
spectral_types = {
    "O": (-0.4, -0.3),
    "B": (-0.3, 0.0),
    "A": (0.0, 0.3),
    "F": (0.3, 0.6),
    "G": (0.6, 1.0),
    "K": (1.0, 1.8),
    "M": (1.8, 3.0),
    "L": (3.0, 5.0),
}

valid = gal_df["bp_rp"].notna() & gal_df["M_G"].notna()
gal_clean = gal_df[valid].copy()
total_clean = len(gal_clean)
print(f"Sources with valid BP-RP and M_G: {total_clean}")
print()

labels, fractions, counts = [], [], []
print(f"{'Type':<6} | {'Count':<8} | {'Fraction':<10}")
print("-" * 30)
for stype, (xmin, xmax) in spectral_types.items():
    mask = (gal_clean["bp_rp"] >= xmin) & (gal_clean["bp_rp"] < xmax)
    n = int(mask.sum())
    frac = 100 * n / total_clean
    labels.append(stype)
    counts.append(n)
    fractions.append(frac)
    print(f"{stype:<6} | {n:<8} | {frac:.2f}%")

print("-" * 30)
print(f"K+M combined: {fractions[labels.index('K')] + fractions[labels.index('M')]:.1f}%")
print(f"G           : {fractions[labels.index('G')]:.1f}%")
print()
print("Paper values to compare against: K 35.1%, M 34.5% (K+M = 69.6%), G 15.2%")

## 6. HR diagram with spectral-type histogram

This reproduces Fig. 10 of the paper. MIST isochrones are optional and require a MIST isochrone/track file (`.iso.cmd`) downloaded separately from the [MIST web interpolator](https://waps.cfa.harvard.edu/MIST/interp_isos.html) — update `iso_file` below to point to your local copy.

In [ ]:
iso_file = "MIST_iso_69148a306bac4.iso.cmd"  # update path as needed
iso_columns = [
    "EEP", "log10_isochrone_age_yr", "initial_mass", "star_mass", "log_Teff", "log_g", "log_L",
    "FeH_init", "FeH", "Bessell_U", "Bessell_B", "Bessell_V", "Bessell_R", "Bessell_I",
    "2MASS_J", "2MASS_H", "2MASS_Ks", "Kepler_Kp", "Kepler_D51", "Hipparcos_Hp", "Tycho_B", "Tycho_V",
    "Gaia_G_DR2Rev", "Gaia_BP_DR2Rev", "Gaia_RP_DR2Rev", "Gaia_G_MAW", "Gaia_BP_MAWb", "Gaia_BP_MAWf",
    "Gaia_RP_MAW", "TESS", "Gaia_G_EDR3", "Gaia_BP_EDR3", "Gaia_RP_EDR3", "phase",
]

iso = None
if Path(iso_file).exists():
    iso = pd.read_csv(iso_file, delim_whitespace=True, comment="#", names=iso_columns, header=None)
    iso.columns = iso.columns.str.strip()
    iso["BP_RP"] = iso["Gaia_BP_EDR3"] - iso["Gaia_RP_EDR3"]
    iso["Gmag"] = iso["Gaia_G_EDR3"]
    iso = iso.dropna(subset=["BP_RP", "Gmag"])
else:
    print(f"[INFO] Isochrone file '{iso_file}' not found — plotting HR diagram without MIST tracks.")

In [ ]:
# --- Figure 1: HR diagram only ---
fig1, ax = plt.subplots(figsize=(6, 8))

has_simbad = gal_clean["Simbad_ID"].notna()
matched = gal_clean[has_simbad]
unmatched = gal_clean[~has_simbad]

ax.scatter(unmatched["bp_rp"], unmatched["M_G"], s=18, color="lightgray",
           alpha=0.85, edgecolor="none", zorder=5, rasterized=True)
ax.scatter(matched["bp_rp"], matched["M_G"], s=18, color="lightgray",
           alpha=0.85, edgecolor="black", linewidth=0.3, zorder=8)

# Spectral-type bar at the top of the HR diagram
y_top = 15.5
bar_h = 0.5
colors_bar = ["lightgray", "gray"] * len(spectral_types)
for i, (stype, (xmin, xmax)) in enumerate(spectral_types.items()):
    ax.fill_betweenx([y_top, y_top - bar_h], xmin, xmax, color=colors_bar[i], alpha=0.6)
    ax.text((xmin + xmax) / 2, y_top, stype, ha="center", va="bottom", fontsize=9)

# MIST mass tracks (only if isochrone file was found)
if iso is not None:
    masses_to_plot = [0.1, 0.2, 0.5, 1.0, 5.0, 10]
    colors_mass = ["cyan", "lightblue", "blue", "lightgreen", "orange", "red"]
    for mass, color in zip(masses_to_plot, colors_mass):
        iso_mass = iso[np.isclose(iso["star_mass"], mass, rtol=0.03)]
        if not iso_mass.empty:
            iso_mass_sorted = iso_mass.sort_values("EEP")
            ax.plot(iso_mass_sorted["BP_RP"], iso_mass_sorted["Gmag"], lw=1.6,
                    color=color, label=f"{mass} M☉", alpha=1, zorder=24)
    ax.legend(markerscale=2, fontsize="small", loc="center right")

ax.invert_yaxis()
ax.set_xlabel("BP − RP [mag]")
ax.set_ylabel(r"Absolute $M_G$ [mag]")
ax.set_xlim(-0.8, 5.1)
ax.set_ylim(17, -7.5)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("hr_diagram_galactic_sources.pdf", dpi=300, bbox_inches="tight")
plt.savefig("hr_diagram_galactic_sources.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"SIMBAD matches plotted: {has_simbad.sum()} / {len(gal_clean)}")




In [ ]:
# --- Age-isochrone HR diagram ---
ages_to_plot = [1e6,1e7, 1e8, 5e8, 1e9, 5e9,1e10]  # 10 Myr, 100 Myr, 500 Myr, 1 Gyr, 5 Gyr
cmap = cm.get_cmap("rainbow", len(ages_to_plot))

fig, ax = plt.subplots(figsize=(6, 8))

# --- Gaia stars, styled like the main HR diagram (SIMBAD-matched highlighted) ---
has_simbad = gal_clean["Simbad_ID"].notna()
matched = gal_clean[has_simbad]
unmatched = gal_clean[~has_simbad]

ax.scatter(unmatched["bp_rp"], unmatched["M_G"], s=18, color="lightgray",
           alpha=0.85, edgecolor="none", zorder=5, rasterized=True)
ax.scatter(matched["bp_rp"], matched["M_G"], s=18, color="lightgray",
           alpha=0.85, edgecolor="black", linewidth=0.3, zorder=8)

# --- Spectral-type bar at the top, same as the main HR diagram ---
y_top = 15.5
bar_h = 0.5
colors_bar = ["lightgray", "gray"] * len(spectral_types)
for i, (stype, (xmin, xmax)) in enumerate(spectral_types.items()):
    ax.fill_betweenx([y_top, y_top - bar_h], xmin, xmax, color=colors_bar[i], alpha=0.6)
    ax.text((xmin + xmax) / 2, y_top, stype, ha="center", va="bottom", fontsize=9)

# --- Age isochrones ---
for i, age in enumerate(ages_to_plot):
    iso_age = iso[np.isclose(10 ** iso["log10_isochrone_age_yr"], age, rtol=0.05)]
    if iso_age.empty:
        continue
    iso_age_sorted = iso_age.sort_values("EEP")
    ax.plot(iso_age_sorted["BP_RP"], iso_age_sorted["Gmag"], lw=1.6,
            color=cmap(i), alpha=1, label=f"{age/1e9} Gyr", zorder=24)

ax.legend(markerscale=2, fontsize="small", loc="center right", title="Isochrone ages")

# --- Axes, same conventions as the main HR diagram ---
ax.invert_yaxis()
ax.set_xlabel("BP − RP [mag]")
ax.set_ylabel(r"Absolute $M_G$ [mag]")
ax.set_xlim(-0.8, 5.1)
ax.set_ylim(17, -7.5)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("hr_diagram_ages.pdf", dpi=300, bbox_inches="tight")
plt.savefig("hr_diagram_ages.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure 2: spectral-type fraction histogram (equally-sized bars) ---
fig2, ax_hist = plt.subplots(figsize=(6, 4))

bars = ax_hist.bar(labels, fractions, color="lightgray", edgecolor="black")
for bar, frac in zip(bars, fractions):
    ax_hist.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f"{frac:.1f}%", ha="center", va="bottom", fontsize=8)

ax_hist.set_ylabel("Fraction [%]")
ax_hist.set_xlabel("Spectral type")
ax_hist.set_ylim(0, max(fractions) * 1.3)
ax_hist.grid(alpha=0.3)
ax_hist.tick_params(axis="x", labelsize=10)

plt.tight_layout()
plt.savefig("spectral_type_fractions.pdf", dpi=300, bbox_inches="tight")
plt.savefig("spectral_type_fractions.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. SIMBAD OTYPE distribution

In [ ]:
otype_df = (
    gal_df.loc[gal_df["Simbad_ID"].notna(), "Simbad_class"]
    .str.decode("utf-8")
    .str.replace(r"(?i)candidate", "Cand", regex=True)
    .value_counts()
    .reset_index()
)
otype_df.columns = ["OTYPE", "Count"]

top_n = min(39, len(otype_df))
top_classes = otype_df[:top_n]
palette = plt.cm.viridis(np.linspace(0, 1, top_n))

fig, ax = plt.subplots(figsize=(20, 6))
ax.bar(top_classes["OTYPE"], top_classes["Count"], color=palette)
ax.set_title("Top Most Frequent SIMBAD OTYPE Classes", size=17)
ax.set_ylabel("Number of Objects", size=17)
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=90, labelsize=16)
ax.tick_params(axis="y", labelsize=17)
ax.set_yscale("log")
plt.tight_layout()
plt.savefig("simbad_classes_top.pdf")
plt.show()

print(f"SIMBAD-classified sources plotted: {gal_df['Simbad_ID'].notna().sum()}")

## 8. NED Type distribution

In [ ]:
ned_counts_df = gal_df["NED_class"].dropna().value_counts().reset_index()
ned_counts_df.columns = ["NED_class", "Count"]

top_n = min(31, len(ned_counts_df))
top_classes = ned_counts_df[:top_n]
palette = plt.cm.viridis(np.linspace(0, 1, top_n))

fig, ax = plt.subplots(figsize=(20, 5))
ax.bar(top_classes["NED_class"], top_classes["Count"], color=palette)
ax.set_title("NED Type Classes", size=17)
ax.set_ylabel("Number of Objects", size=17)
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=90, labelsize=16)
ax.tick_params(axis="y", labelsize=17)
ax.set_yscale("log")
plt.tight_layout()
plt.savefig("ned_classes.pdf")
plt.show()

# Plot X-ray light curves

In [ ]:
xrb = gal[gal["Simbad_class"]=="HighMassXBin"]
print(len(xrb))

In [ ]:
xrb

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

_warned_approx_time = False  # print the caveat only once per session

# instrument -> (raw LC column, Bayesian-block column, plot color)
INSTRUMENTS = {
    "pn": ("pn_LC", "pn_LC_BB", "red"),
    "M1": ("M1_LC", "M1_LC_BB", "blue"),
    "M2": ("M2_LC", "M2_LC_BB", "green"),
}


def _reconstruct_frame_times(lc_bb):
    """Approximate per-frame times from Bayesian-block start/end times,
    assuming frames are equally spaced within each block."""
    frame_times = []
    for jj in range(lc_bb.shape[0]):
        start, end, n_frames_block = lc_bb[jj, 0], lc_bb[jj, 1], lc_bb[jj, 2]
        n_frames_block = int(n_frames_block) if n_frames_block > 0 else 1
        dt = (end - start) / n_frames_block
        block_times = start + dt * (np.arange(n_frames_block) + 0.5)
        frame_times.append(block_times)
    return np.concatenate(frame_times)


def plot_source_all_observations_published(out, group_id, n_cols=4, save=True):
    """
    Plot the raw light curve + Bayesian-block representation for every
    observation belonging to `group_id`, for all available instruments
    (pn, M1, M2), using ONLY columns available in the published catalogue
    (pn_LC/M1_LC/M2_LC, pn_LC_BB/M1_LC_BB/M2_LC_BB) -- no external cube files.

    Per-frame timestamps are not stored directly in the published table, so
    they are reconstructed from each instrument's own Bayesian-block
    start/end times (frames assumed equally spaced within a block). This is
    an approximation and can be slightly off if excluded/flagged frames
    create gaps not reflected in the block boundaries.

    NOTE: `group_number` in the catalogue reflects the total number of XMM
    observations of this source, including ones that did not pass the
    variability selection (N_blocks >= 3) and are therefore NOT present in
    this (variable-source) catalogue. The number of panels plotted here is
    based on the actual rows found for this group_id in `out`, which can be
    smaller than `group_number`.
    """
    global _warned_approx_time
    if not _warned_approx_time:
        print("[NOTE] Raw light-curve timestamps are reconstructed from "
              "each instrument's Bayesian-block boundaries, since the "
              "published table does not store per-frame times directly. "
              "Times are therefore approximate.")
        _warned_approx_time = True

    group_rows = out[out["group_id"] == group_id]
    if len(group_rows) == 0:
        print(f"No rows found for group_id={group_id}")
        return

    n_obs = len(group_rows)
    n_rows = int(np.ceil(n_obs / n_cols))

    group_number_field = int(np.atleast_1d(group_rows["group_number"])[0])
    if group_number_field != n_obs:
        print(f"[NOTE] group_id={group_id}: catalogue lists group_number="
              f"{group_number_field}, but only {n_obs} observation(s) are "
              f"present in this variable-source catalogue (the rest did not "
              f"pass the variability selection).")

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    axes = np.atleast_1d(axes).flatten()

    plot_idx = 0
    for row in group_rows:
        obsid = row["OBSID"]
        detid = row["DETID"]

        ax = axes[plot_idx]

        # Pick one instrument per observation, priority: pn -> M1 -> M2
        chosen = None
        for inst, (lc_col, bb_col, color) in INSTRUMENTS.items():
            raw_bb = np.ma.filled(np.atleast_2d(row[bb_col]), np.nan)
            valid_bb = ~np.isnan(raw_bb).any(axis=1)
            lc_bb = raw_bb[valid_bb]

            raw_lc = np.ma.filled(np.atleast_2d(row[lc_col]), np.nan)
            mask_lc = ~np.isnan(raw_lc[:, 0])
            lc = raw_lc[mask_lc]

            if lc.size == 0 or lc_bb.size == 0:
                continue

            chosen = (inst, lc, lc_bb, color)
            break  # stop at the first available instrument (pn first)

        if chosen is None:
            ax.axis('off')
            plot_idx += 1
            continue

        inst, lc, lc_bb, color = chosen
        lc_bb = lc_bb[np.argsort(lc_bb[:, 0])]
        frame_times = _reconstruct_frame_times(lc_bb)

        n_use = min(len(frame_times), len(lc))
        frame_times = frame_times[:n_use]
        lc = lc[:n_use]

        t0 = lc_bb[0, 0]
        time_raw = frame_times - t0

        # --- Raw light curve: background + source ---
        ax.scatter(time_raw, lc[:, 1], color='gray', alpha=0.6,
                   zorder=1, marker="+", s=10)
        ax.plot(time_raw, lc[:, 0], color=color, alpha=0.5, drawstyle='steps-mid',
                lw=1, zorder=1)
        ax.scatter(time_raw, lc[:, 0], color=color, alpha=0.8,
                   zorder=1, marker="*", s=10)

        # --- Bayesian block steps ---
        rates_bb = lc_bb[:, 3] / np.where(lc_bb[:, 2] == 0, 1, lc_bb[:, 2])
        for jj in range(lc_bb.shape[0]):
            start = lc_bb[jj, 0] - t0
            end = lc_bb[jj, 1] - t0
            rate = rates_bb[jj]
            is_sig = lc_bb[jj, 5] == 1

            ax.plot([start, end], [rate, rate], color='black' if is_sig else 'orange',
                    lw=1.5, solid_capstyle='butt', zorder=3)
            if is_sig:
                ax.axvspan(start, end, color='gray', alpha=0.1, zorder=2)

        # group_id printed inside the panel instead of a figure-level suptitle
        ax.text(0.97, 0.95, f"grp {group_id}", transform=ax.transAxes,
                ha="right", va="top", fontsize=7, color="black",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7))

        if plot_idx % n_cols == 0:
            ax.set_ylabel("Counts", fontsize=10)
        ax.set_xlabel(r"Time $-$ $t_0$ [s]", fontsize=9)
        ax.set_title(f"{detid}\n{obsid} ({inst})", fontsize=9)
        ax.tick_params(labelsize=8)

        plot_idx += 1

    for i in range(plot_idx, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()

    if save:
        plt.savefig(f"lc_group_{group_id}.pdf", bbox_inches='tight')
    plt.show()

In [ ]:
HighMassXBin = gal[gal["Simbad_class"] == "HighMassXBin"]
print(len(HighMassXBin))
print(HighMassXBin[0:5])
for gid in HighMassXBin["group_id"]:
    plot_source_all_observations_published(out, group_id=gid)

In [ ]:
CataclyV = gal[gal["Simbad_class"] == "CataclyV*"]
print(len(CataclyV))
print(CataclyV[0:5])
for gid in CataclyV["group_id"][0:5]:
    plot_source_all_observations_published(out, group_id=gid)